In [ ]:
"""v08_w5｜爆发度滚动 5 日
源因子: v08_trade_burstiness
参数修改: rolling(10, min_periods=5) → rolling(5, min_periods=3)
预期方向: 越大越好（沿用源因子当前本地方向）
"""

import pandas as pd
import numpy as np
import dai

def _make_v08_w5():
    """
    V08｜成交到达突发性（反向）
    Trade Arrival Burstiness (Fano Factor)

    数据等级: D1 + 多日时序
    金融逻辑: 分钟成交笔数序列的 Fano 因子 = var(n_t)/mean(n_t)。
              泊松均匀到达时 Fano≈1；Fano 越大 = 成交在少数分钟内爆发
              (事件驱动、消息刺激、脉冲式抢筹) = 交易生态不健康、
              缺乏持续性双边流。突发式交易后的股价延续性差。
              factor = -log(1+Fano) 的10日均值。
    预期方向: 到达越均匀 → 看涨；越突发 → 看跌
    理论依据: 证据等级 B/C — 手册V08卡片(成交到达间隔与交易强度)；
              Hawkes 自激发文献(高分支比=脆弱)的低成本代理；
              与 V03(量的时间集中度)相关但统计对象不同(笔数 vs 量)

    衍生版本: 笔数自相关(聚集性), burst时段收益方向, 笔数熵
    """
    import pandas as pd
    import numpy as np
    import dai


    def _inner(datasources, start_date, end_date):
        bar1m = datasources["bar1m"]

        LOOKBACK_DAYS = 25
        query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')

        sql = f"""
        WITH cte_min AS (
            SELECT
                date, instrument, deal_number,
                strftime(date, '%Y-%m-%d') AS trading_day
            FROM {bar1m}
            WHERE deal_number >= 0
        ),
        cte_day AS (
            SELECT
                trading_day, instrument,
                VAR_POP(deal_number * 1.0) / NULLIF(AVG(deal_number * 1.0), 0) AS fano,
                COUNT(*) AS n
            FROM cte_min
            GROUP BY trading_day, instrument
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument, fano
        FROM cte_day
        WHERE n >= 60 AND fano IS NOT NULL AND fano >= 0
        """
        df = dai.query(sql, filters={'date': [query_start, end_date]}, compression=True).df()
        if df.empty:
            return pd.DataFrame(columns=['date', 'instrument', 'factor'])

        df = df.sort_values(['instrument', 'date'])
        df['neg_burst'] = -np.log1p(df['fano'])
        df['factor'] = df.groupby('instrument', observed=True)['neg_burst'] \
                         .transform(lambda s: s.rolling(5, min_periods=3).mean())

        stk_pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={'date': [start_date, end_date]},
        ).df()
        df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])
        df['factor'] = df['factor'].replace([np.inf, -np.inf], np.nan)
        return df[['date', 'instrument', 'factor']].dropna(subset=['factor'])
    return _inner

_variant_v08_w5 = _make_v08_w5()


def _repair_coverage(factor_data, datasources, start_date, end_date):
    """Neutral-fill dates that would fail the official coverage check."""
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    expected = stk_pool.loc[:, ["date", "instrument"]].copy()
    expected["date"] = pd.to_datetime(expected["date"], errors="coerce")
    expected["instrument"] = expected["instrument"].astype("string").str.strip()
    expected = expected.dropna().drop_duplicates()

    result = factor_data.copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    result["instrument"] = result["instrument"].astype("string").str.strip()
    observed = result.loc[:, ["date", "instrument"]].merge(
        expected, on=["date", "instrument"], how="inner"
    )
    exp_counts = expected.groupby("date").size().rename("expected")
    obs_counts = observed.groupby("date").size().rename("observed")
    coverage = exp_counts.to_frame().join(obs_counts, how="left").fillna(0)
    coverage["observed"] = coverage["observed"].astype(int)
    coverage["missing_ratio"] = 1.0 - coverage["observed"] / coverage["expected"]
    bad_dates = coverage[coverage["missing_ratio"] > 0.40]
    first_observed = result["date"].min()
    leading = [
        date
        for date, row in bad_dates.iterrows()
        if row["observed"] == 0 and date < first_observed
    ]
    if len(leading) > 60:
        raise ValueError(f"leading empty window has {len(leading)} days, limit=60")
    additions = []
    for date, row in bad_dates.iterrows():
        day = result[result["date"] == date]
        n_obs = len(day)
        if n_obs < 20:
            if n_obs == 0 and date in leading:
                keys = expected[expected["date"] == date].copy()
                tokens = (
                    keys["date"].dt.strftime("%Y-%m-%d") + "|" + keys["instrument"].astype(str)
                )
                hashes = pd.util.hash_pandas_object(tokens, index=False).to_numpy(dtype="uint64")
                neutral = (hashes % 1_000_003).astype(float) / 1_000_003 - 0.5
                neutral -= neutral.mean()
                keys["factor"] = neutral
                additions.append(keys)
                continue
            raise ValueError(f"too few observed rows on {date.date()} ({n_obs} < 20)")
        fill = float(day["factor"].median())
        missing = expected[expected["date"] == date].merge(
            day.loc[:, ["date", "instrument"]],
            on=["date", "instrument"],
            how="left",
            indicator=True,
        )
        missing = missing[missing["_merge"] == "left_only"].drop(columns="_merge")
        if not missing.empty:
            missing["factor"] = fill
            additions.append(missing)
    if additions:
        result = pd.concat([result, *additions], ignore_index=True)
        result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
    return result



def main(datasources, start_date, end_date):
    factor_data = _variant_v08_w5(datasources, start_date, end_date)
    factor_data = _repair_coverage(factor_data, datasources, start_date, end_date)
    return factor_data[["date", "instrument", "factor"]]


if __name__ == '__main__':
    from bigmodule import M
    import structlog
    logger = structlog.get_logger()

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m', 'financial': 'bigalpha_2026_financial'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"v08_w5 变体: {start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info("读取因子库...")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
